# Early Exit Comparison: SimCLR vs Supervised
Loads saved results from both early exit notebooks and compares accuracy, exit rates, and efficiency across thresholds.

**Run `early_exit_simclr.ipynb` and `early_exit_supervised.ipynb` first** to generate the `.json` and `.pt` result files.

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import models
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from pathlib import Path
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE)

with open('simclr_ee_results.json') as f:
    simclr_results = json.load(f)
with open('sup_ee_results.json') as f:
    sup_results = json.load(f)

print('SimCLR results loaded:', len(simclr_results), 'thresholds')
print('Supervised results loaded:', len(sup_results), 'thresholds')

Device: mps


FileNotFoundError: [Errno 2] No such file or directory: 'sup_ee_results.json'

### Accuracy vs Threshold

In [ ]:
taus        = [x[0] for x in simclr_results]
simclr_accs = [x[1] for x in simclr_results]
sup_accs    = [x[1] for x in sup_results]

plt.figure(figsize=(7, 4))
plt.plot(taus, simclr_accs, 'o-', label='SimCLR', color='steelblue')
plt.plot(taus, sup_accs,    's-', label='Supervised', color='tomato')
plt.xlabel('Confidence Threshold (tau)')
plt.ylabel('Routed Accuracy')
plt.title('Routed Accuracy vs Threshold')
plt.legend(); plt.tight_layout()
plt.savefig('compare_accuracy.png', dpi=150); plt.show()

### Early Exit Rate vs Threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, results, name, color in [
    (axes[0], simclr_results, 'SimCLR',     'steelblue'),
    (axes[1], sup_results,    'Supervised', 'tomato'),
]:
    taus = [x[0] for x in results]
    ax.plot(taus, [x[2] for x in results], 'o-', label='Exit 1 (layer2)', color='green')
    ax.plot(taus, [x[3] for x in results], 's-', label='Exit 2 (layer3)', color='orange')
    ax.plot(taus, [x[4] for x in results], '^-', label='Final (layer4)',  color=color)
    ax.set_xlabel('Threshold tau'); ax.set_ylabel('Exit Rate')
    ax.set_title(f'{name} — Exit Rates vs Threshold')
    ax.legend()

plt.tight_layout(); plt.savefig('compare_exit_rates.png', dpi=150); plt.show()

### Summary Table (tau=0.90)

In [ ]:
import pandas as pd

rows = []
for results, name in [(simclr_results, 'SimCLR'), (sup_results, 'Supervised')]:
    # Find tau=0.90
    r = next(x for x in results if abs(x[0] - 0.90) < 0.001)
    rows.append({
        'Model':         name,
        'Accuracy':      f'{r[1]:.4f}',
        'Exit1 Rate':    f'{r[2]:.4f}',
        'Exit2 Rate':    f'{r[3]:.4f}',
        'Final Rate':    f'{r[4]:.4f}',
        'Early Exit %':  f'{(r[2]+r[3])*100:.1f}%',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

### Accuracy vs Early Exit Rate (efficiency frontier)

In [ ]:
plt.figure(figsize=(7, 4))

for results, name, color, marker in [
    (simclr_results, 'SimCLR',     'steelblue', 'o'),
    (sup_results,    'Supervised', 'tomato',    's'),
]:
    early_pcts = [(x[2]+x[3])*100 for x in results]
    accs       = [x[1] for x in results]
    taus_lbl   = [x[0] for x in results]
    plt.plot(early_pcts, accs, marker+'-', color=color, label=name)
    for ep, ac, tau in zip(early_pcts, accs, taus_lbl):
        plt.annotate(f'τ={tau}', (ep, ac), textcoords='offset points', xytext=(4, 3), fontsize=7)

plt.xlabel('Early Exit Rate (% samples not reaching final layer)')
plt.ylabel('Routed Accuracy')
plt.title('Accuracy vs Efficiency Trade-off')
plt.legend(); plt.tight_layout()
plt.savefig('compare_frontier.png', dpi=150); plt.show()

### Reload Models and Run Side-by-Side on Test Set

In [ ]:
test_dir = Path('dataset_ood/test')
val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_ds     = ImageFolder(test_dir, transform=val_tfms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
num_classes = len(test_ds.classes)
print('Classes:', test_ds.classes)

In [ ]:
def build_backbone_and_heads(num_classes, device):
    backbone   = models.resnet18(weights=None)
    backbone.fc = nn.Identity()
    backbone    = backbone.to(device)
    exit1_head = nn.Sequential(nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(), nn.Linear(128, num_classes)).to(device)
    exit2_head = nn.Sequential(nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(), nn.Linear(256, num_classes)).to(device)
    final_head = nn.Linear(512, num_classes).to(device)
    return backbone, exit1_head, exit2_head, final_head

def load_ee_checkpoint(path, num_classes, device):
    bb, e1, e2, ef = build_backbone_and_heads(num_classes, device)
    ckpt = torch.load(path, map_location=device)
    bb.load_state_dict(ckpt['backbone'])
    e1.load_state_dict(ckpt['exit1_head'])
    e2.load_state_dict(ckpt['exit2_head'])
    ef.load_state_dict(ckpt['final_head'])
    for m in [bb, e1, e2, ef]: m.eval()
    return bb, e1, e2, ef

def make_forward(bb):
    def forward_early_exit(x):
        x = bb.conv1(x); x = bb.bn1(x); x = bb.relu(x); x = bb.maxpool(x)
        x = bb.layer1(x); x = bb.layer2(x); e1 = None  # filled below
        return x
    return forward_early_exit

def evaluate_model(backbone, exit1_head, exit2_head, final_head, dataloader, tau1=0.90, tau2=0.90):
    for m in [backbone, exit1_head, exit2_head, final_head]: m.eval()
    total = correct = c1 = c2 = cf = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            x = backbone.conv1(inputs); x = backbone.bn1(x); x = backbone.relu(x); x = backbone.maxpool(x)
            x = backbone.layer1(x)
            x = backbone.layer2(x); e1_logits = exit1_head(x)
            x = backbone.layer3(x); e2_logits = exit2_head(x)
            x = backbone.layer4(x); x = backbone.avgpool(x); x = torch.flatten(x, 1)
            ef_logits = final_head(x)

            p1 = torch.softmax(e1_logits, 1); conf1, pred1 = p1.max(1)
            p2 = torch.softmax(e2_logits, 1); conf2, pred2 = p2.max(1)
            predf = ef_logits.argmax(1)

            batch_preds = []
            for i in range(labels.size(0)):
                if   conf1[i] >= tau1: batch_preds.append(pred1[i].item()); c1 += 1
                elif conf2[i] >= tau2: batch_preds.append(pred2[i].item()); c2 += 1
                else:                  batch_preds.append(predf[i].item()); cf += 1

            bt = torch.tensor(batch_preds, device=labels.device)
            correct += (bt == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(batch_preds)
            all_labels.extend(labels.cpu().numpy())

    return correct/total, c1/total, c2/total, cf/total, all_preds, all_labels

# Load both models
bb_s, e1_s, e2_s, ef_s = load_ee_checkpoint('early_exit_simclr.pt',     num_classes, DEVICE)
bb_p, e1_p, e2_p, ef_p = load_ee_checkpoint('early_exit_supervised.pt', num_classes, DEVICE)
print('Both models loaded.')

In [ ]:
TAU = 0.90
acc_s, r1_s, r2_s, rf_s, preds_s, labels_s = evaluate_model(bb_s, e1_s, e2_s, ef_s, test_loader, TAU, TAU)
acc_p, r1_p, r2_p, rf_p, preds_p, labels_p = evaluate_model(bb_p, e1_p, e2_p, ef_p, test_loader, TAU, TAU)

print(f'\nSimCLR     — acc={acc_s:.4f} | exit1={r1_s:.4f} | exit2={r2_s:.4f} | final={rf_s:.4f}')
print(f'Supervised — acc={acc_p:.4f} | exit1={r1_p:.4f} | exit2={r2_p:.4f} | final={rf_p:.4f}')

### Side-by-Side Confusion Matrices (tau=0.90)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, labels, name in [
    (axes[0], preds_s, labels_s, 'SimCLR (Routed)'),
    (axes[1], preds_p, labels_p, 'Supervised (Routed)'),
]:
    cm = confusion_matrix(labels, preds)
    ConfusionMatrixDisplay(cm, display_labels=test_ds.classes).plot(cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.savefig('compare_confusion.png', dpi=150); plt.show()